<a href="https://colab.research.google.com/github/Piro-D/CreditRiskClassificationProject/blob/model-training/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Piro-D/CreditRiskClassificationProject.git
%cd /content/CreditRiskClassificationProject

Cloning into 'CreditRiskClassificationProject'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 82 (delta 23), reused 75 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.98 MiB | 5.08 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/CreditRiskClassificationProject


In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from src.preprocessing_pipeline import build_preprocessor

X_train = pd.read_csv("Dataset/processed/X_train.csv")
y_train = pd.read_csv("Dataset/processed/y_train.csv").squeeze()
X_test = pd.read_csv("Dataset/processed/X_test.csv")
y_test = pd.read_csv("Dataset/processed/y_test.csv").squeeze()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Naive Bayes": GaussianNB(),
    "K-Nearest Neighbor": KNeighborsClassifier(n_neighbors=5),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision_bad": make_scorer(precision_score, pos_label=0),
    "recall_bad": make_scorer(recall_score, pos_label=0),
    "f1_bad": make_scorer(f1_score, pos_label=0),
}

cv_results = []
test_results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", model)
    ])

    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    cv_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision (Bad)": scores["test_precision_bad"].mean(),
        "CV Recall (Bad)": scores["test_recall_bad"].mean(),
        "CV F1-Score (Bad)": scores["test_f1_bad"].mean(),
    })

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    print(f"================ {name} ================")
    print("Confusion Matrix (Rows: Actual [Bad=0, Good=1], Cols: Pred [Bad=0, Good=1]):")
    print(pd.DataFrame(cm, index=["Actual Bad (0)", "Actual Good (1)"], columns=["Pred Bad (0)", "Pred Good (1)"]))
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["Bad (0)", "Good (1)"], digits=4))

    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Bad=0)": precision_score(y_test, y_pred, pos_label=0),
        "Recall (Bad=0)": recall_score(y_test, y_pred, pos_label=0),
        "F1-Score (Bad=0)": f1_score(y_test, y_pred, pos_label=0),
    })

print("\n--- 5-Fold Cross-Validation Comparison ---")
display(pd.DataFrame(cv_results).sort_values(by="CV F1-Score (Bad)", ascending=False).round(4))

print("\n--- Test Set Comparison ---")
display(pd.DataFrame(test_results).sort_values(by="F1-Score (Bad=0)", ascending=False).round(4))

================ Logistic Regression ================
Confusion Matrix (Rows: Actual [Bad=0, Good=1], Cols: Pred [Bad=0, Good=1]):
                 Pred Bad (0)  Pred Good (1)
Actual Bad (0)             43             47
Actual Good (1)            21            189

Detailed Classification Report:
              precision    recall  f1-score   support

     Bad (0)     0.6719    0.4778    0.5584        90
    Good (1)     0.8008    0.9000    0.8475       210

    accuracy                         0.7733       300
   macro avg     0.7364    0.6889    0.7030       300
weighted avg     0.7622    0.7733    0.7608       300

================ Random Forest ================
Confusion Matrix (Rows: Actual [Bad=0, Good=1], Cols: Pred [Bad=0, Good=1]):
                 Pred Bad (0)  Pred Good (1)
Actual Bad (0)             39             51
Actual Good (1)            10            200

Detailed Classification Report:
              precision    recall  f1-score   support

     Bad (0)     0.7959   

,Model,CV Accuracy,CV Precision (Bad),CV Recall (Bad),CV F1-Score (Bad)
0,Logistic Regression,0.7500,0.6039,0.4762,0.5315
2,Naive Bayes,0.4757,0.3537,0.9000,0.5075
4,Gradient Boosting,0.7514,0.6278,0.4286,0.5062
1,Random Forest,0.7614,0.6756,0.3952,0.4971
3,K-Nearest Neighbor,0.7271,0.5729,0.3619,0.4382



--- Test Set Comparison ---


,Model,Accuracy,Precision (Bad=0),Recall (Bad=0),F1-Score (Bad=0)
4,Gradient Boosting,0.7833,0.7119,0.4667,0.5638
1,Random Forest,0.7967,0.7959,0.4333,0.5612
0,Logistic Regression,0.7733,0.6719,0.4778,0.5584
2,Naive Bayes,0.4333,0.3261,0.8333,0.4688
3,K-Nearest Neighbor,0.7400,0.6200,0.3444,0.4429
